# Seminar: Ranking for RAG — Google Colab Version

Pipeline: BM25 → Dense → RRF → MMR → Cross-encoder rerank (ONNXRuntime INT8)

- Dataset via Hugging Face (BeIR/msmarco subsets)
- Optimized for Google Colab (GPU support, reduced data sizes)
- Outputs under /content/seminar-ranking-starter/runs/{TEAM}/{PIPELINE}/...
- Leaderboard scans local runs and ranks by ndcg@10 (tie-breakers mrr@10, precision@5)

**Для запуска в Google Colab:**
1. Включите GPU: Runtime → Change runtime type → GPU
2. Запустите все ячейки по порядку
3. Результаты будут сохранены в /content/seminar-ranking-starter/


In [ ]:
# Config: team/pipeline, quick-run toggles, paths
TEAM_NAME = 'team_colab'
PIPELINE_NAME = 'baseline_full'
QUICK_RUN = True   # set False for larger run (not recommended in Colab free tier)
MAX_CORPUS = 5000 if QUICK_RUN else 20000  # Reduced for Colab
MAX_QUERIES = 200 if QUICK_RUN else 1000   # Reduced for Colab
SEED = 42

from pathlib import Path
import os

# Google Colab paths
BASE = Path('/content/seminar-ranking-starter')
DATA_DIR = BASE/'dataset'/'data'
RUNS_DIR = BASE/'runs'/TEAM_NAME/PIPELINE_NAME
for d in [DATA_DIR, RUNS_DIR]: d.mkdir(parents=True, exist_ok=True)

print('Running in Google Colab')
print('DATA_DIR =', DATA_DIR)
print('RUNS_DIR =', RUNS_DIR)
print('GPU available:', os.system('nvidia-smi > /dev/null 2>&1') == 0)

Running in Google Colab
DATA_DIR = /content/seminar-ranking-starter/dataset/data
RUNS_DIR = /content/seminar-ranking-starter/runs/team_colab/baseline_full
GPU available: True


In [ ]:
# Environment setup for Google Colab
import sys, subprocess

# Install required packages for Colab
print('Installing required packages for Google Colab...')
pkgs = [
    'numpy','pandas','tqdm','datasets','huggingface_hub','sentence-transformers',
    'faiss-cpu','rank-bm25','onnx','onnxruntime','optimum[onnxruntime]','torch',
    # lzma compatibility
    # 'backports.lzma','lzmaffi','cffi'
]

# Install packages with progress
for pkg in pkgs:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable,'-m','pip','install','-q', pkg])

print('All packages installed successfully!')

Installing required packages for Google Colab...
Installing numpy...
Installing pandas...
Installing tqdm...
Installing datasets...
Installing huggingface_hub...
Installing sentence-transformers...
Installing faiss-cpu...
Installing rank-bm25...
Installing onnx...
Installing onnxruntime...
Installing optimum[onnxruntime]...
Installing torch...
All packages installed successfully!


In [ ]:
# Dependency check (prints missing packages and suggests a pip command)
import importlib, sys
to_check = [
    ('numpy', 'numpy'),
    ('pandas', 'pandas'),
    ('tqdm', 'tqdm'),
    ('datasets', 'datasets'),
    ('huggingface_hub', 'huggingface_hub'),
    ('sentence-transformers', 'sentence_transformers'),
    ('rank-bm25', 'rank_bm25'),
    ('faiss-cpu', 'faiss'),
    ('onnxruntime', 'onnxruntime'),
    ('optimum[onnxruntime]', 'optimum.onnxruntime'),
    ('torch', 'torch'),
    ('backports.lzma', 'backports.lzma'),
    ('lzmaffi', 'lzmaffi'),
    ('cffi', 'cffi'),
]
missing = []
for pkg, mod in to_check:
    try:
        importlib.import_module(mod)
        print(f'[OK] {pkg}')
    except Exception as e:
        print(f'[MISS] {pkg}: {e}')
        missing.append(pkg)
if missing:
    pip_cmd = 'pip install ' + ' '.join(f'"{p}"' if '[' in p else p for p in missing)
    print('\nInstall missing deps (terminal recommended):')
    print(pip_cmd)
else:
    print('All dependencies present.')

[OK] numpy
[OK] pandas
[OK] tqdm
[OK] datasets
[OK] huggingface_hub
[OK] sentence-transformers
[OK] rank-bm25
[OK] faiss-cpu
[OK] onnxruntime
[OK] optimum[onnxruntime]
[OK] torch
[OK] backports.lzma
[OK] lzmaffi
[OK] cffi
All dependencies present.


In [ ]:
# Imports and helpers
import os, json, math, time, random, statistics, re
import numpy as np
import pandas as pd
from tqdm import tqdm
random.seed(SEED); np.random.seed(SEED)

TOKEN_SPLIT = re.compile(r"[^\w]+", flags=re.UNICODE)
def tokenize(text, lowercase=True):
    if lowercase and isinstance(text, str):
        text = text.lower()
    # simple regex split baseline
    return [t for t in TOKEN_SPLIT.split(text) if t]

In [ ]:
tokenize("how are you")

['how', 'are', 'you']

In [ ]:
# # Patch for Python without stdlib lzma support
# try:
#     import lzma  # type: ignore
# except ModuleNotFoundError:
#     import sys
#     try:
#         import backports.lzma as lzma  # type: ignore
#         sys.modules['lzma'] = lzma
#         print('Patched lzma via backports.lzma')
#     except Exception as e:
#         print('Failed to patch lzma:', e)

## Google Colab Setup
**Важно для Google Colab:**
- Данные будут загружены в `/content/seminar-ranking-starter/`
- При перезапуске runtime данные сохранятся в Google Drive (опционально)
- Для экономии времени используйте QUICK_RUN=True


## Dataset via Hugging Face (BeIR/fiqa)
Writes unified schema under dataset/data:
- corpus.jsonl
- queries.eval.jsonl
- qrels.eval.tsv


In [ ]:
!pip install ir_datasets

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.9 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=9c8e3c05a51ecfd71725b9351aee3d41ac2efaf9f2ea07b94e01b76b0416b3dc
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  Created wheel for cbor: filename=cbor-1.0.0-cp312-cp312-linux_x86_64.whl size=55024 sha256=0c3a23a7f50184ceccda76db10b5bcff5f37257e2043ec7db43c93b74b1dd8af
  Stored in directory: /root/.cache/pip/wheels/44/3e/21/a739cbcc331a1ab45c326d6edbdac6118de4402f6076e30ff1
Successfully built warc3-wet-clueweb09 cbor


In [ ]:
# =========================
# FIQA via ir_datasets (beir/fiqa/test) → unified files
# =========================
# Цель:
# - Избежать нестабильностей поиска файлов в снапшоте и "None" в идентификаторах.
# - Надёжно получить:
#     corpus.jsonl            # {"docid": str, "text": str, "title": str (может быть пустой)}
#     queries.eval.jsonl      # {"qid": str, "text": str}
#     qrels.eval.tsv          # qid \t docid \t rel
# - Применить QUICK_RUN caps: MAX_QUERIES и MAX_CORPUS.
#
# Источник: ir_datasets (https://ir-datasets.com/)
#   dataset = ir_datasets.load("beir/fiqa/test")
#   - queries_iter(): namedtuple<query_id, text>
#   - qrels_iter():   namedtuple<query_id, doc_id, relevance, iteration>
#   - docs_iter():    namedtuple<doc_id, text, title?>  (если не доступен на split, попробуем base "beir/fiqa")

import json, random, os
from pathlib import Path

try:
    import ir_datasets
except Exception:
    raise RuntimeError('Install ir_datasets first: pip install ir_datasets')

corpus_out = DATA_DIR/'corpus.jsonl'
queries_out = DATA_DIR/'queries.eval.jsonl'
qrels_out = DATA_DIR/'qrels.eval.tsv'

# ---- helper: safe string/id checks ----
INVALID_ID_TOKENS = {'', 'none', 'null', 'nan'}

def is_valid_id(v) -> bool:
    if not isinstance(v, (str, int)):
        return False
    s = str(v).strip().lower()
    return s not in INVALID_ID_TOKENS

def peek_file(path: Path, n=2, label=''):
    print(f'[CHECK]{label}', path)
    try:
        with open(path, 'r', encoding='utf-8') as f:
            for i in range(n):
                line=f.readline()
                if not line: break
                print(' ', line.strip())
    except Exception as e:
        print('  [ERR]', e)

# ---- load dataset ----
ds = ir_datasets.load("beir/fiqa/test")

# ---- step A: collect queries FIRST, cap by MAX_QUERIES ----
# Это гарантирует, что мы не останемся без запросов.
queries_all = []
for q in ds.queries_iter():  # namedtuple<query_id, text>
    if is_valid_id(q.query_id) and isinstance(q.text, str) and q.text.strip():
        queries_all.append((str(q.query_id), q.text.strip()))

random.Random(SEED).shuffle(queries_all)
queries_sel = queries_all[:MAX_QUERIES]
sel_qids = set(q for q,_ in queries_sel)
print(f'[INFO] Queries (loaded/selected): {len(queries_all)}/{len(queries_sel)}')

# ---- step B: collect qrels only for selected qids ----
# qrels_iter(): namedtuple<query_id, doc_id, relevance, iteration>
qrels_sel_all = []
needed_docids = set()
for r in ds.qrels_iter():
    if r.query_id in sel_qids:
        # relevance может быть int/float; приведём >0 к 1, иначе 0
        try:
            rel_num = float(r.relevance)
            rel_bin = 1 if rel_num > 0 else 0
        except Exception:
            # на всякий случай пропускаем странные значения
            continue
        qrels_sel_all.append((str(r.query_id), str(r.doc_id), rel_bin))
        if rel_bin > 0:
            needed_docids.add(str(r.doc_id))

print(f'[INFO] Qrels selected: {len(qrels_sel_all)} (unique docids with rel>0: {len(needed_docids)})')

# ---- step C: collect docs (prefer only those referenced by qrels), cap by MAX_CORPUS ----
# Обычно docs_iter доступен на split; если нет — берём из базового "beir/fiqa".
def iter_docs(source):
    for d in source.docs_iter():
        # в FIQA обычно есть doc_id, text, title?
        did = getattr(d, 'doc_id', None)
        txt = getattr(d, 'text', '')
        title = getattr(d, 'title', '') or ''
        if is_valid_id(did) and (txt or title):
            yield (str(did), str(txt), str(title))

# Attempt docs from split; fallback to base dataset if needed
docs_source = ds
try:
    # быстрый тест доступа к итератору
    _ = ds.docs_count()  # не у всех датасетов, но вызов даст понять наличие
except Exception:
    # docs_count может отсутствовать; не критично — просто будем итерировать
    pass
try:
    # проверим наличие атрибута docs_iter
    getattr(ds, 'docs_iter')
except AttributeError:
    docs_source = ir_datasets.load("beir/fiqa")
    print('[INFO] docs_iter not on split, using base "beir/fiqa".')

# Если хотим минимизировать корпус — попробуем брать сначала только нужные docids;
# однако docs_iter идёт по всем документам; отфильтруем по needed_docids и остановимся по MAX_CORPUS.
corpus_rows = []
corpus_docids_out = set()
n_docs_written = 0
for did, text, title in iter_docs(docs_source):
    if needed_docids and (did not in needed_docids):
        continue
    corpus_rows.append({'docid': did, 'text': text, 'title': title})
    corpus_docids_out.add(did)
    n_docs_written += 1
    if n_docs_written >= MAX_CORPUS:
        break

# Если needed_docids пуст (например, qrels все 0, либо их нет) — соберём просто первый MAX_CORPUS документов.
if n_docs_written == 0:
    for did, text, title in iter_docs(docs_source):
        corpus_rows.append({'docid': did, 'text': text, 'title': title})
        corpus_docids_out.add(did)
        n_docs_written += 1
        if n_docs_written >= MAX_CORPUS:
            break

print(f'[INFO] Corpus written: {n_docs_written}')

# ---- step D: write queries.eval.jsonl ----
queries_rows = [{'qid': qid, 'text': text} for qid, text in queries_sel]

# ---- step E: write qrels.eval.tsv intersected with present corpus docids ----
qrels_rows = []
for qid, did, rel in qrels_sel_all:
    if did in corpus_docids_out:
        qrels_rows.append((qid, did, int(rel)))

print(f'[INFO] Qrels rows after doc intersection: {len(qrels_rows)}')

# ---- step F: write files ----
def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def write_tsv(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for qid, did, rel in rows:
            f.write(f'{qid}\t{did}\t{rel}\n')

write_jsonl(corpus_out, corpus_rows)
write_jsonl(queries_out, queries_rows)
write_tsv(qrels_out, qrels_rows)

# ---- step G: sanity check (no None ids) ----
def has_none_ids(path: Path, key: str) -> bool:
    cnt, bad = 0, 0
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            cnt += 1
            o = json.loads(line)
            v = o.get(key)
            if not is_valid_id(v):
                bad += 1
                if bad <= 2:
                    print('[BAD]', o)
    if bad:
        print(f'[WARN] {bad}/{cnt} rows with invalid {key} in {path}')
        return True
    return False

_ = has_none_ids(corpus_out, 'docid')
_ = has_none_ids(queries_out, 'qid')

# ---- summaries & peeks ----
print('[WRITE]', corpus_out, f'(docs={len(corpus_rows)})')
print('[WRITE]', queries_out, f'(q={len(queries_rows)})')
print('[WRITE]', qrels_out, f'(rows={len(qrels_rows)})')
peek_file(corpus_out, 2, label=' corpus:')
peek_file(queries_out, 2, label=' queries:')
print('FIQA (ir_datasets) unified files are ready.')


[INFO] [starting] opening zip file
[INFO] If you have a local copy of https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/17918ed23cd04fb15047f73e6c3bd9d9
[INFO] [starting] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip
[INFO] [finished] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip: [00:08] [17.9MB] [2.00MB/s]
[INFO] [finished] opening zip file [10.62s]
[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]
[INFO] [starting] building docstore


[INFO] Queries (loaded/selected): 648/200
[INFO] Qrels selected: 533 (unique docids with rel>0: 533)


[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]
docs_iter: 100%|██████████████████████| 57638/57638 [00:02<00:00, 20561.62doc/s]
[INFO] [finished] docs_iter: [00:02] [57638doc] [20546.78doc/s]
[INFO] [finished] building docstore [2.81s]


[INFO] Corpus written: 532
[INFO] Qrels rows after doc intersection: 532
[WRITE] /content/seminar-ranking-starter/dataset/data/corpus.jsonl (docs=532)
[WRITE] /content/seminar-ranking-starter/dataset/data/queries.eval.jsonl (q=200)
[WRITE] /content/seminar-ranking-starter/dataset/data/qrels.eval.tsv (rows=532)
[CHECK] corpus: /content/seminar-ranking-starter/dataset/data/corpus.jsonl
  {"docid": "470", "text": "\"Based on the conversations in the comments, I believe a pragmatic solution would be the best immediate course of action, while still working on the long term addiction issues. The first step is to get your husband to agree to give you all of his credit cards and let you manage the money for a set period of time, say 3 months, to see how it goes. (In my experience people are more likely to agree to being uncomfortable for a finite period of time, rather than indefinitely.) Step 2 is to provide him a means for making purchases on his own, but with a limited budget. Here are some

## BM25 retrieval
Build BM25 index and retrieve top-K per query. Saves runs/.../bm25/topK.jsonl and latency.csv.


In [ ]:
# =========================
# 4) BM25 RETRIEVAL (ROBUST + VALIDATION)
# =========================
# Что делает ячейка:
# - Надёжно загружает корпус и запросы из унифицированных файлов.
# - Валидирует записи и отбрасывает невалидные (без docid/qid/текста).
# - Токенизирует корпус/запросы.
# - Строит BM25Okapi и достаёт top-K.
# - Записывает topK.jsonl и latency.csv.
# - Печатает несколько примеров для проверки (чтобы не было None/null в JSONL).

import re, json, csv, time
import numpy as np
from tqdm import tqdm
from rank_bm25 import BM25Okapi

bm25_dir = RUNS_DIR/'bm25'
bm25_dir.mkdir(parents=True, exist_ok=True)
TOP_K = 100

TOKEN_SPLIT = re.compile(r"[^\w]+", flags=re.UNICODE)

def tokenize(text: str, lowercase: bool = True):
    "Простая лексическая токенизация: split по не-словесным символам"
    if not isinstance(text, str):
        text = '' if text is None else str(text)
    if lowercase:
        text = text.lower()
    return [t for t in TOKEN_SPLIT.split(text) if t]

def read_jsonl(path):
    "Построчное чтение JSONL"
    with open(path, 'r', encoding='utf-8') as f:
        for il, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception as e:
                raise ValueError(f'Bad JSON at {path}:{il}: {e}')

def write_jsonl(path, records):
    "Запись JSONL"
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def write_latency_csv(path, per_query_ms):
    values = [ms for _, ms in per_query_ms]
    p50 = float(np.percentile(values, 50)) if values else 0.0
    p95 = float(np.percentile(values, 95)) if values else 0.0
    with open(path, 'w', encoding='utf-8', newline='') as f:
        w = csv.writer(f)
        w.writerow(['qid', 'latency_ms'])
        for qid, ms in per_query_ms:
            w.writerow([qid, f'{ms:.3f}'])
        w.writerow([])
        w.writerow(['summary', 'value'])
        w.writerow(['p50_ms', f'{p50:.3f}'])
        w.writerow(['p95_ms', f'{p95:.3f}'])

def load_and_validate_corpus(corpus_path):
    """
    Загружает корпус:
    - Требуемые поля: docid (str), text или title (str/пусто)
    - Возвращает:
      docids: [str]
      corpus_tokens: [[str]]
      corpus_text_map: dict(docid -> "title. text")
    """
    docids, corpus_tokens = [], []
    corpus_text_map = {}
    bad = 0
    for o in read_jsonl(corpus_path):
        docid = o.get('docid')
        if not isinstance(docid, str) or not docid:
            bad += 1
            continue
        title = o.get('title') or ''
        text = o.get('text') or ''
        full = (f"{title}. {text}").strip()
        corpus_text_map[docid] = full
        docids.append(docid)
        corpus_tokens.append(tokenize(text))  # BM25 по тексту; title уже в full
    return docids, corpus_tokens, corpus_text_map

def load_and_validate_queries(queries_path):
    """
    Загружает запросы:
    - Требуемые поля: qid (str), text (str/непустая после trim)
    - Возвращает:
      qids: [str]
      queries_tokens: [[str]]
      query_texts: [str]
    """
    qids, queries_tokens, query_texts = [], [], []
    bad = 0
    for o in read_jsonl(queries_path):
        qid = o.get('qid')
        text = (o.get('text') or '').strip()
        if not isinstance(qid, str) or not qid:
            bad += 1
            continue
        if not text:
            # Плохой запрос — пустой текст: пропускаем
            bad += 1
            continue
        qids.append(qid)
        query_texts.append(text)
        queries_tokens.append(tokenize(text))
    return qids, queries_tokens, query_texts

# 1) Загружаем и валидируем входы
docids, corpus_tokens, corpus_text_map = load_and_validate_corpus(corpus_out)
qids, queries_tokens, query_texts = load_and_validate_queries(queries_out)

print(f'[INFO] Corpus valid docs: {len(docids)}')
print(f'[INFO] Queries valid:     {len(qids)}')

# 2) Диагностика: несколько примеров (1-2 из корпуса и запросов)
if docids:
    d0 = docids[0]
    print('[EXAMPLE] docid[0]:', d0)
    print('[EXAMPLE] doc text[0]:', (corpus_text_map[d0][:120] + '...') if corpus_text_map[d0] else '')
if qids:
    print('[EXAMPLE] qid[0]:', qids[0])
    print('[EXAMPLE] query[0]:', query_texts[0])

# 3) Если запросов нет — делаем ранний выход (чтобы не писать None)
if not qids:
    print('[WARN] No valid queries — BM25 will be skipped. Check dataset loader output for queries.eval.jsonl content.')
else:
    # 4) Строим BM25
    bm25 = BM25Okapi(corpus_tokens, k1=1.2, b=0.75)

    # 5) Поиск top-K и замер латентности
    per_query_ms = []
    outputs = []
    for qid, qtok in tqdm(list(zip(qids, queries_tokens)), desc='BM25 retrieving'):
        # Если токены пустые, BM25 вернёт нулевые оценки — обработаем мягко
        t0 = time.perf_counter()
        scores = bm25.get_scores(qtok) if qtok else np.zeros(len(docids), dtype='float32')
        k = min(TOP_K, len(scores))
        if k == 0:
            # Защита от пустого корпуса
            per_query_ms.append((qid, 0.0))
            outputs.append({'qid': qid, 'docids': []})
            continue
        idx = np.argpartition(scores, -k)[-k:]
        top_idx = idx[np.argsort(np.array(scores)[idx])[::-1]]
        top_docids = [docids[i] for i in top_idx]
        t1 = time.perf_counter()
        per_query_ms.append((qid, (t1 - t0) * 1000.0))
        outputs.append({'qid': qid, 'docids': top_docids})

    # 6) Пишем результаты и latency
    topk_path = bm25_dir / f'top{TOP_K}.jsonl'
    write_jsonl(topk_path, outputs)
    write_latency_csv(bm25_dir/'latency.csv', per_query_ms)

    # 7) Проверяем первые 2 строки результата, чтобы убедиться, что нет None
    print('[CHECK] First 2 BM25 lines:')
    with open(topk_path, 'r', encoding='utf-8') as f:
        for _ in range(2):
            line = f.readline().strip()
            if not line:
                break
            print(' ', line)
    print('BM25 written to', bm25_dir)


[INFO] Corpus valid docs: 532
[INFO] Queries valid:     200
[EXAMPLE] docid[0]: 470
[EXAMPLE] doc text[0]: . "Based on the conversations in the comments, I believe a pragmatic solution would be the best immediate course of acti...
[EXAMPLE] qid[0]: 4265
[EXAMPLE] query[0]: Does it make any sense to directly contribute to reducing the US national debt?


BM25 retrieving: 100%|██████████| 200/200 [00:00<00:00, 817.05it/s]

[CHECK] First 2 BM25 lines:
  {"qid": "4265", "docids": ["157553", "392163", "293531", "93881", "140947", "79363", "273187", "349710", "496064", "574327", "163048", "163353", "575729", "562896", "183869", "546568", "439459", "417840", "341399", "363591", "273761", "387141", "402240", "111815", "540806", "23217", "169893", "114303", "389028", "159936", "418034", "457294", "51491", "66834", "98112", "2860", "268026", "561377", "545421", "79807", "296528", "231727", "356161", "200603", "147765", "329662", "147646", "156640", "304284", "197151", "388646", "407401", "278699", "61864", "478514", "187039", "374956", "164008", "122485", "67676", "424598", "108302", "146632", "110848", "198349", "274870", "28230", "517299", "344041", "228341", "399406", "31483", "287157", "437100", "521095", "487067", "51873", "12232", "420529", "41625", "520395", "450694", "28119", "176596", "395912", "69523", "33990", "472585", "558703", "306144", "530570", "273282", "225395", "505678", "44256", "345697", "15

## Dense retrieval with FAISS
Encode corpus and queries with all-MiniLM-L6-v2; retrieve top-K using inner product.


In [ ]:
# =========================
# 5) DENSE RETRIEVAL WITH FAISS
# =========================
# What this cell does:
# - Encodes corpus and queries using sentence-transformers.
# - Builds FAISS inner-product index over corpus vectors.
# - Searches top-K for each query and writes outputs.

from sentence_transformers import SentenceTransformer
import faiss

dense_dir = RUNS_DIR/'dense'; dense_dir.mkdir(parents=True, exist_ok=True)
DENSE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH = 256

def batch_iter(items, bs):
    for i in range(0, len(items), bs):
        yield items[i:i+bs]

def encode_texts(model, texts, batch_size=BATCH, normalize=True):
    vecs = []
    for chunk in tqdm(list(batch_iter(texts, batch_size)), desc='Encoding'):
        v = model.encode(chunk, normalize_embeddings=normalize, show_progress_bar=False)
        vecs.append(v)
    import numpy as np
    return np.vstack(vecs).astype('float32')

# Prepare corpus and queries (text already loaded in previous cell)
corpus_texts = [doc for doc in (corpus_text_map[d] for d in docids)]
model = SentenceTransformer(DENSE_MODEL)

# Encode
doc_embs = encode_texts(model, corpus_texts, batch_size=BATCH, normalize=True)
q_embs = encode_texts(model, query_texts, batch_size=BATCH, normalize=True)

# FAISS index (inner product)
index = faiss.IndexFlatIP(doc_embs.shape[1])
index.add(doc_embs)

# Search
import numpy as np
D, I = index.search(q_embs, TOP_K)
dense_outputs = []
for qi, qid in enumerate(qids):
    top_docids = [docids[j] for j in I[qi]]
    dense_outputs.append({'qid': qid, 'docids': top_docids})

write_jsonl(dense_dir/f'top{TOP_K}.jsonl', dense_outputs)
print('Dense written to', dense_dir)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding: 100%|██████████| 1/1 [00:00<00:00, 17.52it/s]

Dense written to /content/seminar-ranking-starter/runs/team_colab/baseline_full/dense


## RRF fusion
Fuse BM25 and Dense rankings. Parameter rrf.k controls damping (default 60).


In [ ]:
# =========================
# 6) RRF FUSION (BM25 + DENSE)
# =========================
# RRF (Reciprocal Rank Fusion):
# score(d) = sum_over_rankers 1/(k + rank(d))
# We fuse BM25 and Dense lists into a single ranking.

rrf_dir = RUNS_DIR/'fusion_rrf'; rrf_dir.mkdir(parents=True, exist_ok=True)
RRF_K = 60

def load_rankmap(path):
    "Return dict: qid -> [docids]"
    r = {}
    for o in read_jsonl(path):
        r[o['qid']] = o['docids']
    return r

bm25_map = load_rankmap(bm25_dir/f'top{TOP_K}.jsonl')
dense_map = load_rankmap(dense_dir/f'top{TOP_K}.jsonl')

def rrf_fuse_two(list_a, list_b, k=60, top_k=TOP_K):
    scores = {}
    for i, d in enumerate(list_a, start=1):
        scores[d] = scores.get(d, 0.0) + 1.0/(k+i)
    for i, d in enumerate(list_b, start=1):
        scores[d] = scores.get(d, 0.0) + 1.0/(k+i)
    ranked = [d for d,_ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]]
    return ranked

fused_outputs = []
for qid in qids:
    bm = bm25_map.get(qid, [])
    dn = dense_map.get(qid, [])
    fused = rrf_fuse_two(bm, dn, k=RRF_K, top_k=TOP_K)
    fused_outputs.append({'qid': qid, 'docids': fused})

write_jsonl(rrf_dir/f'top{TOP_K}.jsonl', fused_outputs)
print('RRF written to', rrf_dir)


RRF written to /content/seminar-ranking-starter/runs/team_colab/baseline_full/fusion_rrf


## MMR diversification
Select top-10 diverse set from top-20 fused candidates using embeddings.


In [ ]:
# =========================
# 7) MMR DIVERSIFICATION
# =========================
# Maximal Marginal Relevance (MMR):
# score = λ * sim(query, doc) - (1-λ) * max_sim(doc, selected)
# We apply MMR to RRF fused candidates to obtain a top-10 diversified list.

mmr_dir = RUNS_DIR/'mmr'; mmr_dir.mkdir(parents=True, exist_ok=True)
MMR_LAMBDA = 0.5
MMR_SELECT_K = 100
CANDIDATE_POOL = 20

# Map docid -> index in doc_embs
doc_index = {d:i for i,d in enumerate(docids)}

def mmr_select(query_vec, cand_docids, doc_embs, lam=0.5, k=10):
    import numpy as np
    selected = []
    remaining = cand_docids.copy()
    while remaining and len(selected) < k:
        best_d, best_score = None, -1e9
        for d in remaining:
            i = doc_index.get(d)
            if i is None:
                continue
            sim = float(np.dot(query_vec, doc_embs[i]))
            div = 0.0
            if selected:
                sims = [float(np.dot(doc_embs[i], doc_embs[doc_index[s]])) for s in selected if s in doc_index]
                div = max(sims) if sims else 0.0
            score = lam * sim - (1.0-lam) * div
            if score > best_score:
                best_score, best_d = score, d
        selected.append(best_d)
        remaining.remove(best_d)
    return selected

# Apply MMR per query
import numpy as np
mmr_outputs = []
for qi, qid in enumerate(qids):
    cands = fused_outputs[qi]['docids'][:CANDIDATE_POOL]
    qv = q_embs[qi]
    mmr_list = mmr_select(qv, cands, doc_embs, lam=MMR_LAMBDA, k=MMR_SELECT_K)
    mmr_outputs.append({'qid': qid, 'docids': mmr_list})

write_jsonl(mmr_dir/f'top{MMR_SELECT_K}.jsonl', mmr_outputs)
print('MMR written to', mmr_dir)


MMR written to /content/seminar-ranking-starter/runs/team_colab/baseline_full/mmr


## Cross-encoder rerank (mandatory) via ONNXRuntime INT8
Rerank top-100 fused candidates.


In [ ]:
# =========================
# 8) RERANK (CROSS-ENCODER) — SIMPLE CUDA CHECK
# =========================
# Что делает эта ячейка:
# - Если torch.cuda.is_available() → используем GPU.
# - Иначе CPU.
# - Используем CrossEncoder из sentence-transformers (без ONNX/ORT).
# - Реренк кандидатов после RRF и сохраняем: runs/.../rerank/topK.jsonl и latency.csv.

import torch, time, json, csv, numpy as np
from pathlib import Path
from sentence_transformers import CrossEncoder

assert 'RUNS_DIR' in globals() and 'TOP_K' in globals(), 'RUNS_DIR/TOP_K not defined'
assert 'qids' in globals() and 'query_texts' in globals() and 'corpus_text_map' in globals(), \
    'Expected qids/query_texts/corpus_text_map from previous cells.'

rerank_dir = RUNS_DIR/'rerank'
rerank_dir.mkdir(parents=True, exist_ok=True)

# Модель кросс-энкодера
RERANK_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
MAX_LEN = 256

# Загрузка fused топ-K
rrf_dir = RUNS_DIR/'fusion_rrf'
fused_path = rrf_dir/f'top{TOP_K}.jsonl'
fused_list = []
with open(fused_path, 'r', encoding='utf-8') as f:
    for line in f:
        fused_list.append(json.loads(line))
assert len(fused_list) == len(qids), 'fused_list and qids length mismatch'

# Выбор устройства
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = (device == 'cuda')
print(f'[RERANK] device={device}')

# Размер батча
RERANK_BATCH = 128 if device == 'cuda' else 32

# Инициализация CrossEncoder
ce = CrossEncoder(RERANK_MODEL, device=device, max_length=MAX_LEN)
# Можно включить fp16 на GPU для ускорения; при проблемах — закомментируйте строку ниже
if device == 'cuda':
    try:
        ce.model.half()
    except Exception:
        pass  # если не поддерживается half, просто игнорируем

def batched(xs, bs):
    for i in range(0, len(xs), bs):
        yield xs[i:i+bs]

per_query_ms = []
rerank_records = []

for qi, qid in enumerate(qids):
    cands = fused_list[qi]['docids'][:TOP_K]
    pairs = [(query_texts[qi], corpus_text_map[d]) for d in cands]

    t0 = time.perf_counter()
    scores = []
    # predict может принимать весь список, но батчим для стабильности памяти
    for chunk in batched(pairs, RERANK_BATCH):
        s = ce.predict(chunk, batch_size=len(chunk), show_progress_bar=False, convert_to_numpy=True)
        # CrossEncoder возвращает бОльшие — лучше (логиты релевантности)
        scores.extend(list(s))
    t1 = time.perf_counter()

    per_query_ms.append((qid, (t1 - t0) * 1000.0))
    order = np.argsort(np.array(scores))[::-1]
    ranked = [cands[i] for i in order]
    rerank_records.append({'qid': qid, 'docids': ranked})

# Сохраняем результаты
with open(rerank_dir/f'top{TOP_K}.jsonl','w',encoding='utf-8') as f:
    for r in rerank_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

# Метрики латентности
vals = [ms for _, ms in per_query_ms]
p50 = float(np.percentile(vals, 50)) if vals else 0.0
p95 = float(np.percentile(vals, 95)) if vals else 0.0
with open(rerank_dir/'latency.csv','w',encoding='utf-8',newline='') as f:
    w = csv.writer(f)
    w.writerow(['qid','latency_ms'])
    for qid, ms in per_query_ms:
        w.writerow([qid, f'{ms:.3f}'])
    w.writerow([])
    w.writerow(['summary','value'])
    w.writerow(['device', device])
    w.writerow(['batch', RERANK_BATCH])
    w.writerow(['p50_ms', f'{p50:.3f}'])
    w.writerow(['p95_ms', f'{p95:.3f}'])

print(f'[RERANK] device={device}; batch={RERANK_BATCH}; p50={p50:.2f} ms; p95={p95:.2f} ms')
print('Rerank written to', rerank_dir)


[RERANK] device=cuda
[RERANK] device=cuda; batch=128; p50=136.25 ms; p95=212.33 ms
Rerank written to /content/seminar-ranking-starter/runs/team_colab/baseline_full/rerank


## Evaluation (P@5/10, Recall@100, MRR@10, NDCG@5/10)
Evaluate reranked outputs against qrels and write metrics.json and per_query.csv.


In [ ]:
# =========================
# 9) EVALUATION (CLASSIC METRICS WITH QRELS)
# =========================
# This mirrors the seminar script logic and writes:
# - metrics.json (aggregates)
# - per_query.csv (per-query metrics)
# Reference: [python.evaluate_all()](seminar-ranking-starter/scripts/metrics_eval.py:218)

import math, csv
from collections import defaultdict

def load_qrels(path: Path):
    qrels = defaultdict(dict)
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line or line.startswith('#'): continue
            parts = line.split('\t') if '\t' in line else line.split()
            if len(parts) < 3: continue
            try:
                rel = int(float(parts[-1]))
            except Exception:
                # non-numeric rel → skip
                continue
            # Support 3-col and TREC 4-col
            if len(parts) >= 4 and parts[1].lower() in {'0','q0'}:
                qid, docid = parts[0], parts[2]
            else:
                qid, docid = parts[0], parts[1]
            qrels[qid][docid] = rel
    return qrels

def precision_at_k(binary_rels, k):
    k=min(k,len(binary_rels)); return 0.0 if k==0 else sum(binary_rels[:k])/float(k)

def recall_at_k(binary_rels,total_rel,k):
    if total_rel<=0: return 0.0
    k=min(k,len(binary_rels)); return sum(binary_rels[:k])/float(total_rel)

def reciprocal_rank_at_k(binary_rels,k):
    k=min(k,len(binary_rels))
    for i in range(k):
        if binary_rels[i]>0: return 1.0/float(i+1)
    return 0.0

def dcg_at_k(graded_rels,k):
    k=min(k,len(graded_rels)); dcg=0.0
    for i in range(k):
        rel=graded_rels[i]; gain=(2.0**rel-1.0); denom=math.log2(i+2); dcg+=gain/denom
    return dcg

def ndcg_at_k(graded_rels,ideal,k):
    a=dcg_at_k(graded_rels,k); b=dcg_at_k(ideal,k); return 0.0 if b<=0 else a/b

def evaluate_all(qrels, preds, p_at=(5,10), r_at=100, mrr_at=10, ndcg_at=(5,10)):
    # Union of qids to ensure missing predictions get zeros
    all_qids = set(qrels.keys()) | set(preds.keys())
    per_q = {}
    zero_rel_q, missing_pred = 0, 0
    for qid in all_qids:
        qrels_for_q = qrels.get(qid, {})
        ranked_docids = preds.get(qid, [])
        if not ranked_docids:
            missing_pred += 1
        total_rel = sum(1 for r in qrels_for_q.values() if r > 0)
        if total_rel == 0:
            zero_rel_q += 1
        graded_seq = [int(qrels_for_q.get(d, 0)) for d in ranked_docids]
        binary_seq = [1 if r>0 else 0 for r in graded_seq]
        ideal_graded = sorted(qrels_for_q.values(), reverse=True)

        p5  = precision_at_k(binary_seq, p_at[0])
        p10 = precision_at_k(binary_seq, p_at[1])
        r100 = recall_at_k(binary_seq, total_rel, r_at)
        mrr10 = reciprocal_rank_at_k(binary_seq, mrr_at)
        ndcg5  = ndcg_at_k(graded_seq, ideal_graded, ndcg_at[0]) if ideal_graded else 0.0
        ndcg10 = ndcg_at_k(graded_seq, ideal_graded, ndcg_at[1]) if ideal_graded else 0.0

        per_q[qid] = {
            'precision@5': p5, 'precision@10': p10, 'recall@100': r100,
            'mrr@10': mrr10, 'ndcg@5': ndcg5, 'ndcg@10': ndcg10,
            'total_relevant': float(total_rel),
        }
    def safe_mean(vs):
        vs=list(vs); return 0.0 if not vs else sum(vs)/float(len(vs))
    agg = {
        'precision@5': safe_mean(m['precision@5'] for m in per_q.values()),
        'precision@10': safe_mean(m['precision@10'] for m in per_q.values()),
        'recall@100': safe_mean(m['recall@100'] for m in per_q.values()),
        'mrr@10': safe_mean(m['mrr@10'] for m in per_q.values()),
        'ndcg@5': safe_mean(m['ndcg@5'] for m in per_q.values()),
        'ndcg@10': safe_mean(m['ndcg@10'] for m in per_q.values()),
        'counts': {
            'total_qids': len(all_qids),
            'qids_with_rels': sum(1 for q in per_q.values() if q['total_relevant']>0),
            'qids_zero_rels': zero_rel_q,
            'qids_missing_predictions': missing_pred,
        },
    }
    return agg, per_q

# Load inputs
qrels = load_qrels(qrels_out)
preds = {o['qid']: o['docids'] for o in read_jsonl(rerank_dir/f'top{TOP_K}.jsonl')}

agg, per_q = evaluate_all(qrels, preds)

# Write
with open(rerank_dir/'metrics.json','w',encoding='utf-8') as f:
    json.dump(agg, f, indent=2, ensure_ascii=False)

with open(rerank_dir/'per_query.csv','w',encoding='utf-8',newline='') as f:
    w=csv.writer(f); w.writerow(['qid','precision@5','precision@10','recall@100','mrr@10','ndcg@5','ndcg@10','total_relevant'])
    for qid,m in sorted(per_q.items()):
        w.writerow([qid, f"{m['precision@5']:.6f}", f"{m['precision@10']:.6f}", f"{m['recall@100']:.6f}",
                    f"{m['mrr@10']:.6f}", f"{m['ndcg@5']:.6f}", f"{m['ndcg@10']:.6f}", int(m['total_relevant'])])

print('[RESULT] Aggregated metrics:', {k: round(v,6) for k,v in agg.items() if k!='counts'})
print('[WRITE]', rerank_dir/'metrics.json')
print('[WRITE]', rerank_dir/'per_query.csv')


[RESULT] Aggregated metrics: {'precision@5': 0.348, 'precision@10': 0.203, 'recall@100': 0.965861, 'mrr@10': 0.811817, 'ndcg@5': 0.709775, 'ndcg@10': 0.734211}
[WRITE] /content/seminar-ranking-starter/runs/team_colab/baseline_full/rerank/metrics.json
[WRITE] /content/seminar-ranking-starter/runs/team_colab/baseline_full/rerank/per_query.csv


## Local leaderboard
Scan runs/*/*/rerank/metrics.json and display top entries.


In [ ]:
# =========================
# 9) FINAL PER-STAGE LEADERBOARD (BM25, DENSE, RRF, MMR, RERANK)
# =========================
# Что делает:
# - Ищет готовые ранки по этапам в RUNS_DIR (bm25, dense, fusion_rrf, mmr, rerank)
# - Считает P@5/10, Recall@100, MRR@10, NDCG@5/10 (градуированно, если есть релевантности)
# - Пишет metrics.json в каждый этап и сводный CSV runs/.../leaderboard_per_stage.csv
# - Показывает таблицу с метриками по этапам, отсортированную по NDCG@10

import json, csv, math, re
from pathlib import Path
from statistics import mean
import pandas as pd

# Путь к qrels; если у вас qrels в памяти, можно вместо чтения файла собрать qrels_map напрямую
qrels_path = BASE/'dataset'/'data'/'qrels.eval.tsv'
assert qrels_path.exists(), f'Qrels not found at: {qrels_path}'

def read_qrels_trec(path: Path):
    """
    Поддерживает 3-кол и TREC 4-кол формат: qid [Q0] docid rel
    Возвращает:
      - rels_map: dict[qid] -> dict[docid] = relevance(int)
      - rels_total: dict[qid] -> количество релевантных (rel > 0)
    """
    rels_map = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.lower().startswith('qid'):
                continue
            parts = line.split()
            if len(parts) == 3:
                qid, docid, rel = parts
            elif len(parts) >= 4:
                qid, _, docid, rel = parts[:4]
            else:
                continue
            try:
                rel = int(rel)
            except:
                # если заголовок или мусор — пропускаем
                continue
            if qid not in rels_map:
                rels_map[qid] = {}
            rels_map[qid][docid] = rel
    rels_total = {qid: sum(1 for _, r in docs.items() if r > 0) for qid, docs in rels_map.items()}
    return rels_map, rels_total

rels_map, rels_total = read_qrels_trec(qrels_path)

def dcg_at_k(rels, k):
    # rels: список релевантностей по порядку ранжирования (int), длиной до k
    dcg = 0.0
    for i, r in enumerate(rels[:k], start=1):
        dcg += (2**r - 1) / math.log2(i + 1)
    return dcg

def ndcg_at_k(ranked_docids, qid, k):
    if qid not in rels_map:
        return 0.0
    gains = [rels_map[qid].get(d, 0) for d in ranked_docids[:k]]
    ideal = sorted(rels_map[qid].values(), reverse=True)
    idcg = dcg_at_k(ideal, k)
    if idcg == 0:
        return 0.0
    return dcg_at_k(gains, k) / idcg

def precision_at_k(ranked_docids, qid, k):
    if qid not in rels_map:
        return 0.0
    relset = {d for d, r in rels_map[qid].items() if r > 0}
    hits = sum(1 for d in ranked_docids[:k] if d in relset)
    return hits / max(1, k)

def recall_at_k(ranked_docids, qid, k):
    if qid not in rels_map:
        return 0.0
    relset = {d for d, r in rels_map[qid].items() if r > 0}
    denom = len(relset)
    if denom == 0:
        return 0.0
    hits = sum(1 for d in ranked_docids[:k] if d in relset)
    return hits / denom

def mrr_at_k(ranked_docids, qid, k):
    if qid not in rels_map:
        return 0.0
    relset = {d for d, r in rels_map[qid].items() if r > 0}
    for i, d in enumerate(ranked_docids[:k], start=1):
        if d in relset:
            return 1.0 / i
    return 0.0

def read_run_jsonl(path: Path):
    # формат строк: {"qid": "...", "docids": ["...", ...]}
    out = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            obj = json.loads(line)
            if 'qid' in obj and 'docids' in obj:
                out.append((str(obj['qid']), list(map(str, obj['docids']))))
    return out

def find_top_file(stage_dir: Path, preferred_top: int):
    # Предпочитаем top{preferred_top}.jsonl, иначе берём largest top*.jsonl
    exact = stage_dir / f'top{preferred_top}.jsonl'
    if exact.exists():
        return exact, preferred_top
    candidates = list(stage_dir.glob('top*.jsonl'))
    if not candidates:
        return None, None
    # выбираем по максимальному K
    best = None
    best_k = -1
    for p in candidates:
        m = re.match(r'top(\d+)\.jsonl$', p.name)
        if m:
            k = int(m.group(1))
            if k > best_k:
                best_k, best = k, p
    return best, best_k

def read_latency(stage_dir: Path):
    # читает p50_ms/p95_ms из latency.csv, если есть
    lat_path = stage_dir / 'latency.csv'
    p50 = None; p95 = None
    if lat_path.exists():
        try:
            with open(lat_path, 'r', encoding='utf-8') as f:
                rows = list(csv.reader(f))
            for r in rows:
                if len(r) >= 2 and r[0] == 'p50_ms':
                    p50 = float(r[1])
                if len(r) >= 2 and r[0] == 'p95_ms':
                    p95 = float(r[1])
        except:
            pass
    return p50, p95

def evaluate_stage(stage_name: str, stage_dir: Path, preferred_top: int):
    run_path, used_k = find_top_file(stage_dir, preferred_top)
    if run_path is None:
        return None
    pairs = read_run_jsonl(run_path)
    # Пересобираем словарь qid -> список docids
    q_to_docs = {qid: docs for qid, docs in pairs if qid in rels_map}
    if not q_to_docs:
        return None

    # Список qid для оценки — пересечение с qrels
    eval_qids = list(q_to_docs.keys())

    # Метрики
    p5 = mean(precision_at_k(q_to_docs[qid], qid, 5) for qid in eval_qids)
    p10 = mean(precision_at_k(q_to_docs[qid], qid, 10) for qid in eval_qids)
    r100 = mean(recall_at_k(q_to_docs[qid], qid, 100) for qid in eval_qids)
    mrr10 = mean(mrr_at_k(q_to_docs[qid], qid, 10) for qid in eval_qids)
    ndcg5 = mean(ndcg_at_k(q_to_docs[qid], qid, 5) for qid in eval_qids)
    ndcg10 = mean(ndcg_at_k(q_to_docs[qid], qid, 10) for qid in eval_qids)

    # Сохраняем metrics.json в каталог этапа
    metrics = {
        'stage': stage_name,
        'used_top_k': used_k,
        'num_queries': len(eval_qids),
        'precision@5': round(p5, 6),
        'precision@10': round(p10, 6),
        'recall@100': round(r100, 6),
        'mrr@10': round(mrr10, 6),
        'ndcg@5': round(ndcg5, 6),
        'ndcg@10': round(ndcg10, 6),
    }
    p50, p95 = read_latency(stage_dir)
    if p50 is not None: metrics['latency_p50_ms'] = round(p50, 3)
    if p95 is not None: metrics['latency_p95_ms'] = round(p95, 3)

    with open(stage_dir/'metrics.json', 'w', encoding='utf-8') as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    return metrics

# Этапы в порядке отображения
stages = [
    ('bm25', RUNS_DIR/'bm25'),
    ('dense', RUNS_DIR/'dense'),
    ('fusion_rrf', RUNS_DIR/'fusion_rrf'),
    ('mmr', RUNS_DIR/'mmr'),
    ('rerank', RUNS_DIR/'rerank'),
]

collected = []
for name, sdir in stages:
    if sdir.exists():
        m = evaluate_stage(name, sdir, preferred_top=TOP_K)
        if m is not None:
            collected.append(m)

assert collected, 'No stage metrics collected — убедитесь, что этапы выполнены и top*.jsonl присутствуют.'

# Сводная таблица и сохранение CSV
df = pd.DataFrame(collected)
df = df.sort_values(['ndcg@10', 'mrr@10', 'precision@5'], ascending=[False, False, False]).reset_index(drop=True)

leaderboard_csv = RUNS_DIR/'leaderboard_per_stage.csv'
df.to_csv(leaderboard_csv, index=False)
print('Per-stage leaderboard written to:', leaderboard_csv)


Per-stage leaderboard written to: /content/seminar-ranking-starter/runs/team_colab/baseline_full/leaderboard_per_stage.csv


In [ ]:
df[::-1]

,stage,used_top_k,num_queries,precision@5,precision@10,recall@100,mrr@10,ndcg@5,ndcg@10,latency_p50_ms,latency_p95_ms
4,bm25,100,200,0.266,0.1600,0.855873,0.641062,0.538435,0.568198,1.116,2.007
3,mmr,100,200,0.231,0.1685,0.839401,0.836962,0.587366,0.650960,NaN,NaN
2,fusion_rrf,100,200,0.336,0.2005,0.965861,0.797623,0.696930,0.729550,NaN,NaN
1,rerank,100,200,0.348,0.2030,0.965861,0.811817,0.709775,0.734211,136.252,212.327
0,dense,100,200,0.378,0.2230,0.978569,0.860000,0.771834,0.801478,NaN,NaN


## Завершение работы в Google Colab

**Результаты сохранены в:**
- `/content/seminar-ranking-starter/runs/team_colab/baseline_full/`
- Метрики: `rerank/metrics.json`
- Лидерборд: `runs/leaderboard.csv`

**Для сохранения результатов:**
1. Установите `SAVE_TO_DRIVE = True` в ячейке загрузки данных
2. Или скачайте файлы через File → Download

**Советы для Google Colab:**
- Используйте GPU для ускорения (Runtime → Change runtime type → GPU)
- QUICK_RUN=True рекомендуется для бесплатного аккаунта
- При перезапуске runtime все данные будут потеряны (кроме Google Drive)
